In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, precision_recall_curve, auc, accuracy_score, precision_score, recall_score, f1_score

In [2]:
data = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/pope/pope_base_des_05_11_2025.csv")

In [3]:
preds = []
for i in data["answer"]:
    pred = i[:10].lower()
    if "yes" in pred:
        preds.append("yes")
    elif "no" in pred:
        preds.append("no")
    else:
        preds.append("unknown")
        
data["prediciton"] = preds
pd.Series(preds).value_counts()

yes    4817
no     4093
Name: count, dtype: int64

In [4]:
data.head(2)

,question,gt_answer,question_id,image_id,image_path,data_type,answer,prediciton
0,Is there a snowboard in the image?,yes,3f4a998f-caf1-428d-baa3-80c014d751e2,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,adversarial,"Yes, there is a snowboard in the image, and th...",yes
1,Is there a backpack in the image?,no,14371c3a-6a62-419f-893c-1ca360490a7b,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,adversarial,"No, there is no backpack in the image. The per...",no


In [5]:
for name, group in data.groupby("data_type"):
    print(f"Type: {name}")
    # f1_scores = f1_score(group["gt_answer"].tolist(), group["prediciton"].tolist())
    acc_scores = accuracy_score(group["gt_answer"].tolist(), group["prediciton"].tolist())
    # print(f"F1 Score: {f1_scores}")
    print(f"Accuracy Score: {acc_scores}")

Type: adversarial
Accuracy Score: 0.7953333333333333
Type: popular
Accuracy Score: 0.8586666666666667
Type: random
Accuracy Score: 0.8917525773195877


In [6]:
accuracy_score(data["gt_answer"], preds)

0.8481481481481481

In [7]:
# tatget-word based evaluation

# tatget-word based evaluation

In [8]:
result_df = pd.read_pickle("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/pope/pope_llava_label_with_evidence_and_attn_detection_05_11_2025.pkl")

In [12]:
result_df.head(2)

,question,answer,question_id,image_id,image_path,gt_answer,data_type,labels_with_evidence,target_word
0,Is there a snowboard in the image?,"Yes, there is a snowboard in the image, and th...",3f4a998f-caf1-428d-baa3-80c014d751e2,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,yes,adversarial,"[{'word': 'Yes,', 'evidence': [0.023237491, 0....",snowboard
1,Is there a backpack in the image?,"No, there is no backpack in the image. The per...",14371c3a-6a62-419f-893c-1ca360490a7b,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,no,adversarial,"[{'word': 'No,', 'evidence': [0.007417327, 0.0...",backpack


In [11]:
result_df["target_word"] = result_df["question"].apply(lambda x: x.replace("Is there a", "").replace("in the image?","").strip())

In [14]:
all_preds = []
for ans in result_df["answer"]:
    pred = ans[:10].lower()
    if "yes" in pred:
        all_preds.append("yes")
    elif "no" in pred:
        all_preds.append("no")
    else:
        all_preds.append("unknown")

In [16]:
pd.Series(all_preds).value_counts()
result_df["pred_label"] = all_preds

In [17]:
result_df.head(2)

,question,answer,question_id,image_id,image_path,gt_answer,data_type,labels_with_evidence,target_word,pred_label
0,Is there a snowboard in the image?,"Yes, there is a snowboard in the image, and th...",3f4a998f-caf1-428d-baa3-80c014d751e2,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,yes,adversarial,"[{'word': 'Yes,', 'evidence': [0.023237491, 0....",snowboard,yes
1,Is there a backpack in the image?,"No, there is no backpack in the image. The per...",14371c3a-6a62-419f-893c-1ca360490a7b,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,no,adversarial,"[{'word': 'No,', 'evidence': [0.007417327, 0.0...",backpack,no


In [175]:
failed_df = result_df[result_df["pred_label"] == result_df["gt_answer"]]

In [176]:
failed_df.head(2)

,question,answer,question_id,image_id,image_path,gt_answer,data_type,labels_with_evidence,target_word,pred_label
0,Is there a snowboard in the image?,"Yes, there is a snowboard in the image, and th...",3f4a998f-caf1-428d-baa3-80c014d751e2,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,yes,adversarial,"[{'word': 'Yes,', 'evidence': [0.023237491, 0....",snowboard,yes
1,Is there a backpack in the image?,"No, there is no backpack in the image. The per...",14371c3a-6a62-419f-893c-1ca360490a7b,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,no,adversarial,"[{'word': 'No,', 'evidence': [0.007417327, 0.0...",backpack,no


In [177]:
failed_df.index = range(len(failed_df))

In [203]:
import torch

faild = []
corrrected_label = []

both_halu = 0
detect_halu = 0
evi_halu = 0
d_res = []
d_res_pred = []
for inx, row in failed_df.iterrows():
    try:
        pred = row["pred_label"]
        t_word = row["target_word"].split(" ")[-1].strip()
        evi_df = pd.DataFrame(row["labels_with_evidence"])
        req_info = evi_df[evi_df["word"] == t_word].iloc[0]
        prob = req_info["label"].item()
        img_evi = (torch.tensor(req_info["evidence"]) >= 0.4).int().sum().item()
        
        if prob <= 0.5 and img_evi <= 1:
            corrrected_label.append("no")
            d_res.append(pred)
            d_res_pred.append("no")
        elif prob > 0.75 and img_evi <= 2:
            new_img_evi = (torch.tensor(req_info["evidence"]) >= 0.3).int().sum().item()
            if new_img_evi > 2:
                corrrected_label.append("yes")
                # d_res_pred.append("yes")
            else:
                corrrected_label.append("no")
                # d_res_pred.append("no")
            # d_res.append(pred)
        elif prob <=  0.75 and img_evi > 2:
            corrrected_label.append("yes")
            # d_res.append(pred)
            # d_res_pred.append("yes")
        else:
            # d_res.append(pred)
            # d_res_pred.append("yes")
            corrrected_label.append("yes")

    except Exception as e:
        faild.append(inx)
        corrrected_label.append(pred)

In [204]:
accuracy_score(failed_df["gt_answer"].tolist(), corrrected_label)

0.7500330819108112

In [205]:
pd.Series(d_res).value_counts()

yes    1393
no        2
Name: count, dtype: int64

In [206]:
pd.Series(d_res_pred).value_counts()

no    1395
Name: count, dtype: int64

In [193]:
df = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/haloc_extension/caption/gemini_labeled_28k.csv")
df_1 = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/haloc_extension/instruct/gemini_labeled_40k.csv")
df_2 = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/haloc_extension/vqa/tp_data.csv")
df_3 = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/haloc_extension/vqa/tn_data.csv")
coco_data = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/hal_detection_head_train_datasets/coco/gemini_labeld_15k.csv")
total_df = pd.concat([df, df_1, df_2, df_3, coco_data])

In [195]:
df_1

,image_id,question,answer,candidates,hallucination_candidates,candidates_inx,hallucination_candidates_inx,image_path,question_id
0,2317351.jpg,Describe the color of the hat.,The hat is blue.,"['blue', 'hat']",['blue'],"{'blue': [(11, 15)], 'hat': [(4, 7)]}","{'blue': [(11, 15)]}",/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,e7f137be-9190-42f0-a1b0-7486e8028fad
1,2317351.jpg,Describe the color of the hat.,The hat is gray.,"['hat', 'gray']",['gray'],"{'hat': [(4, 7)], 'gray': [(11, 15)]}","{'gray': [(11, 15)]}",/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,c9903799-0507-4102-84fe-3dcd587ae9e0
2,2376102.jpg,Identify the item located at the bottom of the...,The surfboard is in the top of the image.,"['surfboard', 'top', 'image']",['top'],"{'surfboard': [(4, 13)], 'top': [(24, 27)], 'i...","{'top': [(24, 27)]}",/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,2b6092e2-7bc6-43a2-bffe-f7963ada7a22
3,2376102.jpg,Identify the item located at the bottom of the...,The surfboard is at the bottom of the image.,"['bottom', 'surfboard', 'image']",[],"{'bottom': [(24, 30)], 'surfboard': [(4, 13)],...",{},/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,72315862-cce0-43e0-96e5-4202a1ca7fa2
4,2317132.jpg,Are there umbrellas scattered on the sand?,"No, there is a boat on the sand.","['sand', 'boat', 'no']",[],"{'sand': [(27, 31)], 'boat': [(15, 19)], 'no':...",{},/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,fad201cd-b973-4bb4-8d0c-5a33e83415b4
...,...,...,...,...,...,...,...,...,...
39985,2325982.jpg,Would I find a computer mouse to the right of ...,"Yes, there is a computer mouses to the right o...","['yes', 'computer', 'chair', 'right', 'mouse']","['computer', 'mouse']","{'yes': [(0, 3)], 'computer': [(16, 24)], 'cha...","{'computer': [(16, 24)], 'mouse': [(25, 30)]}",/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,2d085fbe-d929-488d-b375-04e17d9e8da9
39986,2370940.jpg,Can you see a truck positioned to the left of ...,"No, there is a bus to the left of the scooter.","['bus', 'left', 'scooter', 'no']",[],"{'bus': [(15, 18)], 'left': [(26, 30)], 'scoot...",{},/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,c4c1125c-8bc6-49a3-ab0c-d26155d9a302
39987,2370940.jpg,Can you see a truck positioned to the left of ...,"Yes, there is a truck to the left of the scooter.","['yes', 'truck', 'scooter', 'left']","['yes', 'truck']","{'yes': [(0, 3)], 'truck': [(16, 21)], 'scoote...","{'yes': [(0, 3)], 'truck': [(16, 21)]}",/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,d608743e-b098-42d5-a6eb-6eec2bbb73c5
39988,2364545.jpg,What is the location shown?,It is a sidewalk.,['sidewalk'],[],"{'sidewalk': [(8, 16)]}",{},/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,ecc4efca-e269-4be9-8cb4-1c8e83348063
